# Execution Control (Honest Model Track)

Use this notebook as your control sheet.


## Current Ground Truth

- Pipeline works end-to-end.
- Unconstrained `recent_level_auto` is anchor-dominated.
- Honest evaluation mode is `recent_level_auto_capped` with `--calibration-max-weight 0.8`.
- Goal now is to increase model contribution, not anchor dependency.


## Run Order

1. `00a_real_source_active_stock_audit.ipynb`
2. `02b_portable_feature_ablation.ipynb`
3. `05b_global_model_transfer_only.ipynb`
4. `05c_transfer_diagnostics.ipynb`


## Decision Gates (Go / No-Go)

- Gate A: `auto_capped` must beat `SNAIVE12` on WAPE and RMSE.
- Gate B: family/category diagnostics should not regress badly in weak segments.
- Gate C: capped calibration must remain capped behavior (not hidden anchor-only).

If any gate fails: continue R&D, no production model claim.


In [5]:
from pathlib import Path
import pandas as pd

REPORTS = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports')

naive = pd.read_csv(REPORTS / 'm5_dept_store_summary.csv')
capped = pd.read_csv(REPORTS / 'portable_m5_transfer_auto_capped_summary.csv')

cols = ['model','WAPE','RMSE','Bias','MASE_mean']
display(naive[cols])
display(capped[cols])


,model,WAPE,RMSE,Bias,MASE_mean
0,SNAIVE12,0.145962,4164.534665,-1309.703571,1.276554


,model,WAPE,RMSE,Bias,MASE_mean
0,XGBOOST_recent_level_auto_capped,0.14168,3953.061323,-1440.868546,1.342153


In [6]:
# Quick go/no-go check
n = pd.read_csv(REPORTS / 'm5_dept_store_summary.csv').iloc[0]
c = pd.read_csv(REPORTS / 'portable_m5_transfer_auto_capped_summary.csv').iloc[0]

gate_a = (c['WAPE'] < n['WAPE']) and (c['RMSE'] < n['RMSE'])
print('Gate A (beats naive on WAPE+RMSE):', 'PASS' if gate_a else 'FAIL')
print('Naive WAPE/RMSE:', round(float(n['WAPE']),6), round(float(n['RMSE']),3))
print('Capped WAPE/RMSE:', round(float(c['WAPE']),6), round(float(c['RMSE']),3))


Gate A (beats naive on WAPE+RMSE): PASS
Naive WAPE/RMSE: 0.145962 4164.535
Capped WAPE/RMSE: 0.14168 3953.061


## Execution Policy

- Report both best-score mode (`auto`) and honest mode (`auto_capped`).
- For enterprise messaging, rely on `auto_capped` as model-evidence track.
- Only promote learned-model claim if capped/raw gains widen consistently.
